# Signisa — Phase 1 training (GPU)

Attach the `kaggle_prep` notebook's output as input. Edit only the CONFIG cell.
Trains the cnn_transformer embedder (CE or ArcFace), then runs the signer-independent
verification evaluation and writes metrics_report.md + curriculum_db_trained.json + model.pt.

In [ ]:
# CONFIG — the only cell to edit
LOSS = "ce"            # "ce" | "arcface"
EPOCHS = 60
LR = 1e-3
BATCH_SIZE = 256
TENSORS_DIR = "/kaggle/input/kaggle-prep/tensors"

In [ ]:
!git clone -q https://github.com/dayan-battulga/signisa.git /kaggle/working/signisa-repo
%pip install -q /kaggle/working/signisa-repo

In [ ]:
import pandas as pd
import torch

from signisa.config import Config
from signisa.data import ShardDataset
from signisa.eval import held_out_participants, run_evaluation
from signisa.train import train_model

cfg = Config(loss=LOSS, epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE)
device = "cuda" if torch.cuda.is_available() else "cpu"

index = pd.read_csv(f"{TENSORS_DIR}/index.csv")
val_pids = held_out_participants(index.participant_id, cfg.n_val_participants, cfg.seed)
train_pids = [p for p in sorted(index.participant_id.unique()) if p not in val_pids]
print("val participants:", val_pids)

train_ds = ShardDataset(TENSORS_DIR, augment=True, participants=train_pids)
val_ds = ShardDataset(TENSORS_DIR, participants=val_pids)
model, history = train_model(cfg, train_ds, val_ds, device=device)
print("best val top-1:", max(history["val_top1"]))

In [ ]:
REPO = "/kaggle/working/signisa-repo"
metrics = run_evaluation(
    model, TENSORS_DIR, f"{REPO}/data/meta/curriculum_db.json",
    f"{REPO}/data/meta/training_labels.json", cfg, "/kaggle/working",
    device=device, val_participants=val_pids)
torch.save(model.state_dict(), "/kaggle/working/model.pt")

print(f"TAR@FAR5: {metrics['tar_at_far']:.1%} | closed-set top-1: {metrics['top1_closed_set']:.1%}")
collapsed = [c["members"] for c in metrics["clusters"] if c["collapsed"]]
print("collapsed clusters (kill criterion):", collapsed or "none")